# Thinking in Probabilities

Companion notebook for the [Thinking in Probabilities](https://ml-viz-ruby.vercel.app/courses/probability-statistics/01-thinking-in-probabilities) lesson on ML Viz.

We'll make the abstract definitions concrete: simulate sample spaces, check the axioms by brute force, watch conditional probability "zoom in", and see the law of large numbers drag a running average toward the expectation.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

rng = np.random.default_rng(42)

## Intuition — probability is bookkeeping for uncertainty

Every ML model outputs, or reasons about, probabilities — a class score, a next-token
distribution, a confidence. Before the fancy machinery, the foundations are simple and
worth *seeing* rather than memorizing: an **event** is a subset of outcomes, its
**probability** is a long-run frequency, **conditioning** narrows the world to where
something is known, **expectation** is the long-run average, and **variance** is the
spread around it. This notebook establishes each by brute-force simulation first — run a
process a million times and count — then confirms the closed forms with `scipy.stats`.

## 1. Sample spaces and events

A sample space is just the set of possible outcomes. An event is a subset. Probability of an event = (in the equally-likely case) fraction of outcomes in it — and we can verify by simulation.

In [ ]:
omega = np.array([1, 2, 3, 4, 5, 6])          # sample space of one die roll
event_even = {2, 4, 6}                          # the event "roll is even"

rolls = rng.integers(1, 7, size=100_000)
p_even_sim = np.isin(rolls, list(event_even)).mean()
print(f"P(even) theoretical = {3/6:.3f},  simulated = {p_even_sim:.3f}")

**What to notice:** the simulated `P(even)` lands on `0.500` — with 100k rolls the
counted frequency is indistinguishable from the theoretical `3/6`. That *frequency →
probability* link is what lets us check every claim below by simulation.

## 2. The axioms, checked by brute force

Probabilities are non-negative, sum to 1 over the whole space, and add over disjoint events.

In [ ]:
p = np.array([(rolls == k).mean() for k in omega])
print("per-outcome probabilities:", p.round(3))
print("all non-negative:", (p >= 0).all())
print("sum over sample space:", p.sum().round(6))

# additivity over disjoint events: {1,2} and {5,6} share no outcomes
A, B = {1, 2}, {5, 6}
pA  = np.isin(rolls, list(A)).mean()
pB  = np.isin(rolls, list(B)).mean()
pAB = np.isin(rolls, list(A | B)).mean()
print(f"P(A)+P(B) = {pA+pB:.3f}  vs  P(A ∪ B) = {pAB:.3f}")

**What to notice:** all six per-outcome probabilities are non-negative and sum to `1.0`,
and for the disjoint events `{1,2}` and `{5,6}` the probabilities simply **add**:
`P(A)+P(B) = P(A∪B)`. Those are Kolmogorov's three axioms, confirmed by counting.

## 3. Conditional probability = zooming in

`P(A | B)` keeps only the worlds where B happened, then renormalizes. The product rule `P(A,B) = P(A|B) P(B)` follows immediately.

In [ ]:
# Roll two dice. A = "sum is 8", B = "first die shows 6"
d1 = rng.integers(1, 7, size=500_000)
d2 = rng.integers(1, 7, size=500_000)

A = (d1 + d2) == 8
B = d1 == 6

p_A_given_B = A[B].mean()           # zoom into the B-worlds
p_joint     = (A & B).mean()
p_B         = B.mean()

print(f"P(A|B) = {p_A_given_B:.4f}   (theory: 1/6 = {1/6:.4f})")
print(f"P(A|B)·P(B) = {p_A_given_B * p_B:.4f}  vs  P(A,B) = {p_joint:.4f}")

**What to notice:** restricting to the worlds where the first die is 6, the chance the
sum is 8 becomes `1/6` (only `(6,2)` works out of 6 equally-likely second rolls). And
`P(A|B)·P(B)` reproduces the joint `P(A,B)` — the product rule, verified numerically.

## 2. The library way — closed forms from `scipy.stats`

Simulation *estimates* these quantities; `scipy.stats` gives the **exact** values from
the distribution's formulas. Modeling the die as `scipy.stats.randint(1, 7)` (uniform on
1–6) exposes `.mean()`, `.var()`, `.std()`, `.pmf()`, and more — which we check against a
fresh simulation.

In [ ]:
from scipy import stats

die = stats.randint(1, 7)                 # discrete uniform on {1,...,6}
print(f'scipy: mean = {die.mean()}, var = {die.var():.4f}, std = {die.std():.4f}')

sim = rng.integers(1, 7, size=200_000)
print(f'sim  : mean = {sim.mean():.4f}, var = {sim.var():.4f}')

assert abs(die.mean() - 3.5) < 1e-9, "uniform die mean is 3.5"
assert abs(die.var() - 35/12) < 1e-9, "uniform die variance is 35/12"
print('\nclosed-form mean/variance match the simulation ✓')

**What to notice:** the analytic mean `3.5` and variance `35/12 ≈ 2.917` match the
simulated values to two decimals. In practice you reach for `scipy.stats` (or the
distribution's formula) for exact answers, and use simulation to sanity-check or when no
closed form exists.

## 4. Expectation and the law of large numbers

E[X] = 3.5 for a fair die — a value the die never shows. Watch the running average of rolls converge to it.

In [ ]:
rolls = rng.integers(1, 7, size=10_000)
running_mean = np.cumsum(rolls) / np.arange(1, len(rolls) + 1)

plt.figure(figsize=(9, 4))
plt.plot(running_mean, color="#6366f1", lw=1.5, label="running average of rolls")
plt.axhline(3.5, color="#eab308", ls="--", label="E[X] = 3.5")
plt.xscale("log")
plt.xlabel("number of rolls (log scale)")
plt.ylabel("average value")
plt.title("Law of large numbers: the sample mean finds the expectation")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

**What to notice:** the running average wanders early (small samples are noisy) then
homes in on `E[X] = 3.5` — a value the die never actually shows. This is the **law of
large numbers**: averages become predictable even when individual outcomes aren't, which
is *why* Monte-Carlo estimation works.

## 5. Variance: spread around the center

Compare a fair die with a '"loaded"' die that mostly shows 3 and 4 — same mean, very different spread.

In [ ]:
fair   = rng.integers(1, 7, size=100_000)
loaded = rng.choice([1, 2, 3, 4, 5, 6], p=[.05, .1, .35, .35, .1, .05], size=100_000)

for name, x in [("fair", fair), ("loaded", loaded)]:
    print(f"{name:>6}:  mean = {x.mean():.3f},  variance = {x.var():.3f}")

plt.figure(figsize=(9, 3.5))
for name, x, c in [("fair", fair, "#6366f1"), ("loaded", loaded, "#14b8a6")]:
    vals, counts = np.unique(x, return_counts=True)
    plt.bar(vals + (0.18 if name == "loaded" else -0.18), counts / len(x),
            width=0.34, color=c, label=name)
plt.xlabel("die face")
plt.ylabel("probability")
plt.title("Same expectation (3.5), different variance")
plt.legend()
plt.grid(alpha=0.4, axis="y")
plt.show()

**What to notice:** the fair and loaded dice have the **same mean (3.5)** but the loaded
one clusters tightly around 3–4, giving a much smaller **variance**. Mean tells you the
center; variance tells you how much to trust any single draw — both matter.

## 4. Gotchas & limitations

- **The LLN needs a finite mean.** For a heavy-tailed **Cauchy** distribution the mean
  doesn't exist, and the running average *never* settles — averaging more data doesn't
  help. Don't assume sample means always converge.
- **Monte-Carlo error shrinks slowly** — like `1/√n`. Ten-thousandfold more samples buys
  only ~100× more accuracy, so simulation is a blunt instrument for rare events.
- **`P(A|B)` is undefined when `P(B) = 0`.** You can't condition on something that never
  happens.
- **Disjoint ≠ independent.** Mutually exclusive events are the *opposite* of independent
  (if one happens the other can't) — a common confusion.

In [ ]:
# LLN needs a finite mean: Cauchy has none, so the running mean keeps jumping
cauchy = rng.standard_cauchy(10_000)
rm = np.cumsum(cauchy) / np.arange(1, 10_001)
print('Cauchy running mean at n=100, 1k, 10k:', rm[[99, 999, 9999]].round(2), '-> never settles')

# Monte-Carlo error decays like 1/sqrt(n)
print('\nestimating P(roll a 6) = 1/6:')
for n in [100, 10_000, 1_000_000]:
    est = (rng.integers(1, 7, n) == 6).mean()
    print(f'  n={n:>9}: estimate {est:.4f}, error {abs(est - 1/6):.4f}')

**What to notice:** the Cauchy running mean is still swinging by whole units at n=10,000
— no convergence, because there's no finite mean to converge to. And the P(6) error falls
roughly 10× each time n grows 100× (the `1/√n` law): reliable, but slow.

## Key takeaways

- **Probability = long-run frequency**; simulate-and-count verifies the axioms,
  conditioning, and the product rule without any formulas.
- **Conditioning** zooms into the worlds where `B` holds and renormalizes;
  `P(A,B) = P(A|B)P(B)`.
- **Expectation** is the long-run average (LLN), **variance** the spread; two
  distributions can share a mean yet differ wildly in variance.
- Use `scipy.stats` for exact means/variances; simulate to sanity-check.
- Mind the edges: **finite-mean requirement** for the LLN, **`1/√n`** Monte-Carlo error,
  undefined conditioning on zero-probability events, and disjoint ≠ independent.

**Next:** [Probability Distributions](https://ml-viz-ruby.vercel.app/courses/probability-statistics/02-probability-distributions).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Conditional probability by counting

With a finite, equally-likely sample space, conditioning is *counting in a smaller world*:

$$P(A \mid B) = \frac{|A \cap B|}{|B|}$$

Implement it for events given as collections of outcomes. The checks use two dice — including the lesson's key idea that for **independent** events, conditioning changes nothing.

In [ ]:
def conditional_probability(A, B, space):
    """P(A | B) for events A, B given as collections of equally-likely outcomes."""
    A, B = set(A), set(B)

    # TODO(you): count outcomes in BOTH A and B (hint: set intersection A & B)
    overlap = ...

    # TODO(you): divide by the number of outcomes in B
    return ...

In [ ]:
# Checks — run me
from itertools import product

space = list(product(range(1, 7), repeat=2))   # all 36 two-dice outcomes

sum_is_8 = [o for o in space if o[0] + o[1] == 8]
first_is_3 = [o for o in space if o[0] == 3]
assert abs(conditional_probability(sum_is_8, first_is_3, space) - 1 / 6) < 1e-12, \
    "given first die = 3, only (3,5) of the 6 outcomes sums to 8"

sum_ge_10 = [o for o in space if o[0] + o[1] >= 10]
first_is_6 = [o for o in space if o[0] == 6]
assert abs(conditional_probability(sum_ge_10, first_is_6, space) - 1 / 2) < 1e-12, \
    "given first die = 6, exactly (6,4), (6,5), (6,6) reach 10"

first_even = [o for o in space if o[0] % 2 == 0]
second_odd = [o for o in space if o[1] % 2 == 1]
assert abs(conditional_probability(first_even, second_odd, space) - len(first_even) / len(space)) < 1e-12, \
    "independent events: conditioning changes nothing"

# Edge cases: conditioning on yourself, and disjoint (mutually exclusive) events
assert abs(conditional_probability(sum_is_8, sum_is_8, space) - 1) < 1e-12, \
    "P(A|A) = 1: conditioning on itself is certain"
only_snake_eyes = [o for o in space if o[0] + o[1] == 2]   # only (1, 1)
assert abs(conditional_probability(only_snake_eyes, first_is_6, space)) < 1e-12, \
    "disjoint events: given first die = 6, the sum can never be 2"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def conditional_probability(A, B, space):
    A, B = set(A), set(B)
    overlap = len(A & B)
    return overlap / len(B)
```

</details>

### Exercise 2 — Expectation and variance from a PMF

For a discrete random variable, both summaries are weighted sums over the PMF:

$$\mathbb{E}[X] = \sum_i x_i \, p_i \qquad\qquad \text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

Implement both — use the shortcut formula for the variance, and reuse your `expectation` to get $\mathbb{E}[X^2]$.

In [ ]:
def expectation(values, probs):
    """E[X] = sum of value * probability."""
    values = np.asarray(values, dtype=float)
    probs = np.asarray(probs, dtype=float)

    # TODO(you): the weighted sum
    return ...


def variance(values, probs):
    """Var(X) = E[X^2] - (E[X])^2."""
    values = np.asarray(values, dtype=float)

    # TODO(you): E[X], then E[X^2] (hint: expectation of values ** 2)
    ex = ...
    ex2 = ...

    return ex2 - ex ** 2

In [ ]:
# Checks — run me
die_vals, die_probs = [1, 2, 3, 4, 5, 6], [1 / 6] * 6
assert abs(expectation(die_vals, die_probs) - 3.5) < 1e-12, "fair die: E[X] = 3.5"
assert abs(variance(die_vals, die_probs) - 35 / 12) < 1e-12, "fair die: Var(X) = 35/12"
assert abs(expectation([0, 1], [0.7, 0.3]) - 0.3) < 1e-12, "Bernoulli(0.3): E[X] = p"
assert abs(variance([0, 1], [0.7, 0.3]) - 0.21) < 1e-12, "Bernoulli(0.3): Var = p(1-p)"
assert abs(variance([5, 5], [0.5, 0.5])) < 1e-12, "a constant has zero variance"

# Edge cases: a single-outcome distribution, and a skewed (non 50/50) two-point one
assert abs(expectation([5], [1.0]) - 5) < 1e-12, "a single-outcome distribution: E[X] is just that value"
assert abs(variance([5], [1.0])) < 1e-12, "...and its variance is zero too"
assert abs(variance([0, 10], [0.9, 0.1]) - 9) < 1e-12, \
    "skewed two-point distribution: Var = p(1-p)*(b-a)^2 = 0.09*100 = 9"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def expectation(values, probs):
    values = np.asarray(values, dtype=float)
    probs = np.asarray(probs, dtype=float)
    return np.sum(values * probs)


def variance(values, probs):
    values = np.asarray(values, dtype=float)
    ex = expectation(values, probs)
    ex2 = expectation(values ** 2, probs)
    return ex2 - ex ** 2
```

</details>

---
## Extra practice — descriptive statistics and a binary correlation

Two more DML-OpenProblem exercises that round out "summarizing a sample":

- **`78_descriptive-statistics-calculator`** — given a raw list of numbers
  (not a PMF), compute mean, median, mode, variance, standard deviation, and
  the 25th/50th/75th percentiles + IQR in one pass.
- **`95_calculate-the-phi-coefficient`** — the correlation coefficient
  specialized to two *binary* variables. Build the 2×2 contingency table of
  counts $(n_{11}, n_{10}, n_{01}, n_{00})$ and plug into
  $$\phi = \frac{n_{11}n_{00} - n_{10}n_{01}}{\sqrt{(n_{11}+n_{10})(n_{01}+n_{00})(n_{11}+n_{01})(n_{10}+n_{00})}}$$

In [ ]:
def descriptive_statistics(data):
    """DML 78 — mean/median/mode/variance/std/percentiles/IQR for a raw sample."""
    data = np.asarray(data, dtype=float)

    mean = data.mean()
    median = np.median(data)
    values, counts = np.unique(data, return_counts=True)
    mode = values[np.argmax(counts)]        # smallest value among ties

    # TODO(you): population variance E[(X - mean)^2] and its square root
    variance = ...
    std_dev = ...

    p25, p50, p75 = np.percentile(data, [25, 50, 75])
    # TODO(you): the interquartile range, p75 - p25
    iqr = ...

    return {
        "mean": mean, "median": median, "mode": mode,
        "variance": variance, "standard_deviation": std_dev,
        "25th_percentile": p25, "50th_percentile": p50, "75th_percentile": p75,
        "interquartile_range": iqr,
    }


def phi_corr(x, y):
    """DML 95 — the Phi coefficient (Pearson correlation for two binary variables)."""
    x, y = np.asarray(x), np.asarray(y)
    n11 = np.sum((x == 1) & (y == 1))
    n10 = np.sum((x == 1) & (y == 0))
    n01 = np.sum((x == 0) & (y == 1))
    n00 = np.sum((x == 0) & (y == 0))

    denom = (n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)
    if denom == 0:
        return 0.0   # one variable has zero variance -> correlation undefined, define as 0

    # TODO(you): (n11*n00 - n10*n01) / sqrt(denom)
    return ...

In [ ]:
# Checks — run me

stats_1 = descriptive_statistics([10, 20, 30, 40, 50])
assert abs(stats_1["mean"] - 30.0) < 1e-9 and abs(stats_1["median"] - 30.0) < 1e-9
assert abs(stats_1["mode"] - 10) < 1e-9, "all-unique data: mode falls back to the smallest value"
assert abs(stats_1["variance"] - 200.0) < 1e-6, "DML 78 example"
assert abs(stats_1["standard_deviation"] - 14.142135623730951) < 1e-6
assert abs(stats_1["interquartile_range"] - 20.0) < 1e-9

# Edge case: zero variance (a constant sample)
stats_const = descriptive_statistics([7, 7, 7, 7])
assert abs(stats_const["variance"]) < 1e-12 and abs(stats_const["standard_deviation"]) < 1e-12
assert abs(stats_const["interquartile_range"]) < 1e-12

# Edge case: a genuine tie in the mode
tie = descriptive_statistics([1, 1, 2, 2, 3])
assert abs(tie["mode"] - 1) < 1e-9, "counts tied at 2 for both 1 and 2 -> the smaller value wins"

# 95: DML's own worked example (perfect negative correlation)
assert abs(phi_corr([1, 1, 0, 0], [0, 0, 1, 1]) - (-1.0)) < 1e-9, "DML 95 example"
assert abs(phi_corr([1, 1, 0, 0], [1, 1, 0, 0]) - 1.0) < 1e-9, "identical binary variables -> perfect correlation"

# Edge case: zero variance in x (constant) -> undefined correlation, defined here as 0
assert phi_corr([1, 1, 1, 1], [0, 1, 0, 1]) == 0.0, "a constant variable has no correlation to define"

print("✅ Extra practice passed")

<details>
<summary>💡 Show solution</summary>

```python
def descriptive_statistics(data):
    data = np.asarray(data, dtype=float)
    mean = data.mean()
    median = np.median(data)
    values, counts = np.unique(data, return_counts=True)
    mode = values[np.argmax(counts)]
    variance = ((data - mean) ** 2).mean()
    std_dev = np.sqrt(variance)
    p25, p50, p75 = np.percentile(data, [25, 50, 75])
    iqr = p75 - p25
    return {
        "mean": mean, "median": median, "mode": mode,
        "variance": variance, "standard_deviation": std_dev,
        "25th_percentile": p25, "50th_percentile": p50, "75th_percentile": p75,
        "interquartile_range": iqr,
    }


def phi_corr(x, y):
    x, y = np.asarray(x), np.asarray(y)
    n11 = np.sum((x == 1) & (y == 1))
    n10 = np.sum((x == 1) & (y == 0))
    n01 = np.sum((x == 0) & (y == 1))
    n00 = np.sum((x == 0) & (y == 0))
    denom = (n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)
    if denom == 0:
        return 0.0
    return (n11 * n00 - n10 * n01) / np.sqrt(denom)
```

</details>

**Next:** the [Probability Distributions](https://ml-viz-ruby.vercel.app/courses/probability-statistics/02-probability-distributions) lesson — the four named distributions (and their notebook) that ML reaches for daily.